# Create Datasets with Sampled BERT Embeddings

In [ ]:
# Packages
import pandas as pd
from pathlib import Path
import json
import gc
import numpy as np

## Load Sample Ids

In [1]:
# load sample
# /kaggle/input/datasets/lianestrauch/sample-10/sample_tuning_10per.json

# get sample ids
sample_path = Path("/kaggle/input/datasets/lianestrauch/sample-10/sample_tuning_10per.json")

with open(sample_path, "r") as f:
    sample_info = json.load(f)

sample_ids = pd.DataFrame(sample_info["sample"])

## Load bert-base Embeddings and Filter for the Sample Ids

In [2]:
# load bert-base embeddings
embeddings_path = Path("/kaggle/input/datasets/lianestrauch/bert-base-full/bert-base")
chunk_files = sorted((embeddings_path / "chunks").glob("chunk_*.parquet"))

bert_embeddings = pd.read_parquet(
        chunk_files
    )

# get the embeddings of the sample
sample = bert_embeddings[bert_embeddings.id.isin(set(sample_ids.id))]

print(sample.shape)
sample.head()

(49919, 5)


,id,layer_last,layer_2nd_last,layer_3rd_last,layer_4th_last
27,103374,"[0.15938261151313782, 0.5890845060348511, -0.8...","[0.21944591403007507, 0.914222002029419, -1.18...","[0.32035207748413086, 0.7639493346214294, -0.6...","[0.16036687791347504, 0.9425111413002014, -0.1..."
35,103382,"[-0.2970133423805237, 1.0696569681167603, -0.3...","[-0.16140912473201752, 1.1220661401748657, -0....","[-0.18070949614048004, 0.7860732674598694, -0....","[-0.3338121175765991, 1.2210636138916016, -0.2..."
49,103396,"[0.12266620993614197, 0.6604804992675781, -0.2...","[-0.021101150661706924, 0.5887324213981628, -0...","[0.02516765147447586, 0.4629615843296051, -0.4...","[0.0006751632317900658, 0.3873882591724396, -0..."
50,103397,"[-0.6865489482879639, 0.09895093739032745, -0....","[-0.8765157461166382, 0.14662615954875946, -0....","[-0.851387083530426, -0.023210901767015457, -0...","[-0.868627667427063, 0.3228589594364166, 0.214..."
64,103412,"[-0.19965139031410217, 0.20671319961547852, -0...","[-0.06150959059596062, 0.5854789614677429, -0....","[0.12812305986881256, 0.5802285075187683, -0.2...","[0.14403001964092255, 1.1125648021697998, -0.1..."


In [3]:
# Free up RAM
del bert_embeddings
gc.collect()

0

In [4]:
sample.columns

Index(['id', 'layer_last', 'layer_2nd_last', 'layer_3rd_last',
       'layer_4th_last'],
      dtype='object')

## Create and Store the Mean Embedding for the Last Four Layers

In [5]:
layer_cols = ['layer_last', 'layer_2nd_last', 'layer_3rd_last', 'layer_4th_last']

# Shape: (n_samples, 4, 768)
embeddings = np.stack([
    np.stack(sample[col].to_numpy())
    for col in layer_cols
], axis=1)

# Mean over the 4 layers
# Shape: (n_samples, 768)
mean_embeddings = embeddings.mean(axis=1)

# Create output DataFrame
result = pd.DataFrame({
    "id": sample["id"].values,
    "mean_embedding": list(mean_embeddings)
})

result.to_parquet(
    "sample_bert_base_mean_last4.parquet",
    index=False
)

## Store the Last Layer
(see Apendix G in Mosolova, Candito & Ramisch, 2025)  
Best preforming layer on BERT-base-uncased for Adjectives and Nouns (Clustering: X-Means)

In [6]:
# Create sample for layer 12

# Last layer
sample[["id", "layer_last"]].to_parquet(
    "sample_bert_base_last.parquet",
    index=False
)

## Store the Second to Last Layer
(see Apendix G in Mosolova, Candito & Ramisch, 2025)
best preforming layer on BERT-base-uncased for Adjectives and Nouns (Clustering: AG_silhoutte)

In [7]:
# Create sample for layer 11

# Second to last layer
sample[["id", "layer_2nd_last"]].to_parquet(
    "sample_bert_base_2nd_last.parquet",
    index=False
)